In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [2]:
df = pd.read_csv("data_std_filter.csv")

In [3]:
# predictor variables (same as BM axis variables)
predictors = [
    "prop_car_0",
    "prop_ethnic_white",
    "prop_class_never_worked",
    "prop_dep_4"
]

# dependent variable (same as BM colouring variable)
X = sm.add_constant(df[predictors])
y = df["ahah_pct"]

model = sm.OLS(y, X).fit()

In [4]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               ahah_pct   R-squared:                       0.264
Model:                            OLS   Adj. R-squared:                  0.264
Method:                 Least Squares   F-statistic:                     2873.
Date:                Sun, 19 Apr 2026   Prob (F-statistic):               0.00
Time:                        20:38:55   Log-Likelihood:            -1.4761e+05
No. Observations:               32067   AIC:                         2.952e+05
Df Residuals:                   32062   BIC:                         2.953e+05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                              coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [10]:
#Mediation test
import pandas as pd
import statsmodels.api as sm

df = pd.read_csv("data_std_filter.csv")

def run_ols(y, X_vars, label):
    X = sm.add_constant(df[X_vars])
    model = sm.OLS(df[y], X).fit()
    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"{'='*60}")
    print(f"R² = {model.rsquared:.4f} | F = {model.fvalue:.2f} | N = {int(model.nobs)}")
    print(f"\n{'Variable':<30} {'Coef':>8} {'p>|t|':>8} {'Sig':>5}")
    print("-"*55)
    for var in model.params.index:
        p   = model.pvalues[var]
        sig = "***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "ns"
        print(f"{var:<30} {model.params[var]:>8.3f} {p:>8.4f} {sig:>5}")
    return model

# Total effect: ethnicity on AHAH
m1 = run_ols("ahah_pct", ["prop_ethnic_white"],
             "Total effect: prop_ethnic_white -> ahah_pct")

# Ethnicity on mediator (car deprivation)
m2 = run_ols("prop_car_0", ["prop_ethnic_white"],
             "Ethnicity -> car deprivation (mediator)")

# Mediator on outcome controlling for ethnicity
m3 = run_ols("ahah_pct", ["prop_ethnic_white", "prop_car_0"],
             "Both ethnicity + car deprivation -> ahah_pct")

# Summary
print(f"Ethnicity -> AHAH (total effect):        β = {m1.params['prop_ethnic_white']:.3f}")
print(f"Ethnicity -> Car deprivation:            β = {m2.params['prop_ethnic_white']:.3f}")
print(f"Ethnicity -> AHAH (direct, controlling")
print(f"         for car deprivation):                   β = {m3.params['prop_ethnic_white']:.3f}")
print(f"Car deprivation -> AHAH (controlling")
print(f"         for ethnicity):                         β = {m3.params['prop_car_0']:.3f}")

b1 = m1.params['prop_ethnic_white']
b3 = m3.params['prop_ethnic_white']
indirect = b1 - b3
pct_mediated = (indirect / b1) * 100

print(f"\nTotal effect:                           β = {b1:.3f}")
print(f"Direct effect:                          β = {b3:.3f}")
print(f"Indirect effect (mediation via car deprivation): β = {indirect:.3f}")
print(f"% of total effect mediated by car deprivation:    {pct_mediated:.1f}%")

if abs(b3) < abs(b1) and abs(b3) > 0.01:
    print("\n→ PARTIAL MEDIATION: car deprivation partially mediates")
    print("  the ethnicity-AHAH relationship")
elif abs(b3) < 0.01:
    print("\n→ FULL MEDIATION: car deprivation fully mediates")
    print("  the ethnicity-AHAH relationship")
else:
    print("\n→ NO MEDIATION detected")

# Sobel test for significance of indirect effect
import numpy as np
a     = m2.params['prop_ethnic_white']
b     = m3.params['prop_car_0']
se_a  = m2.bse['prop_ethnic_white']
se_b  = m3.bse['prop_car_0']

sobel_se = np.sqrt(b**2 * se_a**2 + a**2 * se_b**2)
sobel_z  = (a * b) / sobel_se
sobel_p  = 2 * (1 - __import__('scipy').stats.norm.cdf(abs(sobel_z)))

print(f"\nSobel test of indirect effect:")
print(f"  Indirect effect (a×b): {a*b:.4f}")
print(f"  Sobel z-statistic:     {sobel_z:.4f}")
print(f"  p-value:               {sobel_p:.4f}")
print(f"  Significant:           {'Yes ***' if sobel_p < 0.001 else 'Yes **' if sobel_p < 0.01 else 'Yes *' if sobel_p < 0.05 else 'No'}")


Total effect: prop_ethnic_white -> ahah_pct
R² = 0.2005 | F = 8039.84 | N = 32067

Variable                           Coef    p>|t|   Sig
-------------------------------------------------------
const                           106.482   0.0000   ***
prop_ethnic_white                -0.644   0.0000   ***

Ethnicity -> car deprivation (mediator)
R² = 0.2559 | F = 11025.40 | N = 32067

Variable                           Coef    p>|t|   Sig
-------------------------------------------------------
const                            53.163   0.0000   ***
prop_ethnic_white                -0.376   0.0000   ***

Both ethnicity + car deprivation -> ahah_pct
R² = 0.2543 | F = 5466.87 | N = 32067

Variable                           Coef    p>|t|   Sig
-------------------------------------------------------
const                            78.851   0.0000   ***
prop_ethnic_white                -0.448   0.0000   ***
prop_car_0                        0.520   0.0000   ***
Ethnicity -> AHAH (total effect)